In [1]:
import numpy as np
import CI
import CI_physicist

from electron_integrals import *

In [46]:
# Number of orbitals (without spin)
num_orbitals = 8
# Number of electrons
num_electrons = 4
#Include spin?
include_spin = False

num_spin_orbitals = (1+int(include_spin))*num_orbitals

Calculate electron integrals

In [47]:
x_max = 10
num_points = 1000

x = np.linspace(-x_max,x_max,num_points)

pot = GaussianWell(w=100, a=1, center=0)
#pot = HOPotential()

spf, h = get_spf_and_diag_h(num_orbitals, x, pot)

if include_spin:
    #Add spin
    h = np.kron(h, np.eye(2,2))

g = coulomb_interaction_matrix_elements(spf, spf, x, x, kappa = 1, a=0.01)

if include_spin:
    #Add spin
    g = np.kron(g, np.einsum("pr,qs->pqrs",np.eye(2,2), np.eye(2,2)))

g_chemist = g.transpose(0,2,1,3)

In [4]:
%%timeit
H_chemist = CI.AddressHamiltonian(num_spin_orbitals, num_electrons, h, g_chemist).get_hamiltonian()
E_chemist, _ = np.linalg.eigh(H_chemist)
#print(E_chemist)

2 s ± 24.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [5]:
%%timeit
H_physicist = CI_physicist.AddressHamiltonian(num_spin_orbitals, num_electrons, h, g).get_hamiltonian()
E_physicist, _ = np.linalg.eigh(H_physicist)
#print(E_physicist)

1.99 s ± 24 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [6]:
%%timeit
H_SC_physicist = CI_physicist.SlaterCondonHamiltonian(num_spin_orbitals, num_electrons, h, g).get_hamiltonian()
E_SC_physicist, C = np.linalg.eigh(H_SC_physicist)
#print(E_SC_physicist)

305 ms ± 14.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [48]:
H_chemist = CI.AddressHamiltonian(num_spin_orbitals, num_electrons, h, g_chemist).get_hamiltonian()
E_chemist, _ = np.linalg.eigh(H_chemist)
H_physicist = CI_physicist.AddressHamiltonian(num_spin_orbitals, num_electrons, h, g).get_hamiltonian()
E_physicist, _ = np.linalg.eigh(H_physicist)
H_SC_physicist = CI_physicist.SlaterCondonHamiltonian(num_spin_orbitals, num_electrons, h, g).get_hamiltonian()
E_SC_physicist, C = np.linalg.eigh(H_SC_physicist)

In [49]:
np.testing.assert_allclose(E_chemist, E_physicist)
np.testing.assert_allclose(E_chemist, E_SC_physicist)

In [55]:
import DirectCI
import importlib
importlib.reload(DirectCI)
from DirectCI import DirectCI

In [56]:
dCI = DirectCI(h, g, num_orbitals, num_electrons, num_beta_electrons=0)


In [52]:
rng = np.random.default_rng()
C_test = rng.random(len(E_SC_physicist)).astype(np.cdouble)
C_test = np.divide(C_test, np.sqrt(C_test.T.conj()@C_test))
C_test=C_test.reshape(-1,1)

np.testing.assert_allclose(dCI.get_sigma(C_test), H_SC_physicist@C_test)

In [53]:
DirectCI(h, g, num_orbitals, num_electrons).get_sigma_alphabeta(C_test)

array([[4.0212106 +0.j, 3.76326306+0.j, 1.3439741 +0.j, ...,
        1.2969057 +0.j, 1.34915318+0.j, 1.42251421+0.j],
       [4.89319266+0.j, 7.81564903+0.j, 3.72646749+0.j, ...,
        4.09335742+0.j, 3.46713243+0.j, 1.98096134+0.j],
       [5.89962747+0.j, 6.42721912+0.j, 8.95047217+0.j, ...,
        2.94037443+0.j, 5.63383639+0.j, 4.57029206+0.j],
       ...,
       [5.27329279+0.j, 5.63152625+0.j, 4.84147633+0.j, ...,
        7.85411835+0.j, 6.41386684+0.j, 5.65872977+0.j],
       [1.74272701+0.j, 4.00767638+0.j, 4.89688095+0.j, ...,
        2.91569663+0.j, 8.44594005+0.j, 4.06930802+0.j],
       [6.01312136+0.j, 6.08122595+0.j, 6.50169307+0.j, ...,
        6.21869031+0.j, 9.07987958+0.j, 9.20925454+0.j]])

In [68]:
A = np.array([[1,2,3], [1,2,3], [1,1,1]])
#print(A)
print(np.einsum('a, b -> ab', A[0,:], A[1,:]))
#print(np.sum(A[:,0]))

[[1 2 3]
 [2 4 6]
 [3 6 9]]


In [64]:
A[0,:]*A[1,:]

array([1, 4, 9])